In [ ]:
# # CELL 0: Install Required Packages
# # Run this cell once to install all dependencies in a fresh environment.
# # Note: The '!' exclamation mark tells Jupyter to run a shell command.

# !pip install -q pandas numpy scikit-learn xgboost imbalanced-learn
# !pip install -q fastapi uvicorn pydantic
# !pip install -q nest-asyncio joblib
# !pip install -q matplotlib seaborn
# !pip install pandas
# print("All required packages have been successfully installed!")

In [ ]:
# Imports & Environment Setup

import pandas as pd
import numpy as np
import joblib
import json
import time
import os
from pydantic import BaseModel, Field
from typing import Optional
from fastapi import FastAPI, HTTPException
import uvicorn
import nest_asyncio

nest_asyncio.apply()

print(" Libraries Imported Successfully!")
print(" Environment is ready for model integration.")

In [ ]:
# Load Production Assets
import joblib
import os

print("Loading models and assets into memory...")

MODEL_DIR = "production_models"

# Load Stroke Assets
try:
    stroke_model = joblib.load(os.path.join(MODEL_DIR, "champion_stroke_model.pkl"))
    stroke_scaler = joblib.load(os.path.join(MODEL_DIR, "stroke_scaler.pkl"))
    stroke_features = joblib.load(os.path.join(MODEL_DIR, "stroke_expected_features.pkl"))
    print("Stroke assets loaded.")
except Exception as e:
    print(f"Error loading stroke assets: {e}")

# Load Heart Assets
try:
    heart_l0 = joblib.load(os.path.join(MODEL_DIR, "stacking_heart_l0.pkl"))
    heart_meta = joblib.load(os.path.join(MODEL_DIR, "stacking_heart_meta.pkl"))
    heart_scaler = joblib.load(os.path.join(MODEL_DIR, "heart_scaler.pkl"))
    heart_features = joblib.load(os.path.join(MODEL_DIR, "heart_expected_features.pkl"))
    print("Heart assets loaded.")
except Exception as e:
    print(f"Error loading heart assets: {e}")

print("All requested assets loaded successfully.")

In [ ]:
# Unified Data Schema Definition
from pydantic import BaseModel, Field

class PatientData(BaseModel):
    # Demographics and Vitals
    age: float = Field(..., gt=0, lt=120)
    gender: str = Field(..., description="Male, Female, or Other")
    height_cm: float = Field(..., gt=50, lt=250)
    weight_kg: float = Field(..., gt=10, lt=300)

    # Stroke Risk Factors (Medical History)
    hypertension: int = Field(..., ge=0, le=1)
    heart_disease: int = Field(..., ge=0, le=1)
    ever_married: str = Field(..., description="Yes or No")
    work_type: str = Field(..., description="Govt_job, Private, Self-employed, children, Never_worked")
    Residence_type: str = Field(..., description="Urban or Rural")
    smoking_status: str = Field(..., description="formerly smoked, never smoked, smokes, Unknown")

    # Heart Disease Symptoms (Current Status)
    chest_pain_type: int = Field(..., ge=1, le=4, description="1:Typical, 2:Atypical, 3:Non-anginal, 4:Asymptomatic")
    exercise_angina: int = Field(..., ge=0, le=1)

print("Unified PatientData schema defined successfully.")

In [ ]:
# Feature Engineering Engine (Production Ready)

class FeatureEngineer:
    @staticmethod
    def prepare_stroke_data(data: PatientData, expected_cols: list) -> pd.DataFrame:
        # 1. Calculate BMI
        height_m = data.height_cm / 100.0
        bmi = data.weight_kg / (height_m ** 2)
        
        # 2. Categorize BMI
        if bmi <= 18.5: bmi_cat = 'underweight'
        elif bmi <= 25.0: bmi_cat = 'normal'
        elif bmi <= 30.0: bmi_cat = 'overweight'
        else: bmi_cat = 'obese'
            
        # 3. Categorize Age
        if data.age <= 40: age_group = 'young'
        elif data.age <= 55: age_group = 'middle'
        elif data.age <= 65: age_group = 'senior'
        else: age_group = 'elderly'

        # 4. Prepare numerical features
        raw_dict = {
            'age': data.age,
            'hypertension': data.hypertension,
            'heart_disease': data.heart_disease,
            'bmi': bmi,
            'age_x_hypertension': data.age * data.hypertension,
            'age_x_heart_disease': data.age * data.heart_disease,
            'dual_vascular_risk': 1 if (data.hypertension == 1 and data.heart_disease == 1) else 0
        }
        
        # 5. Create a strictly formatted DataFrame filled with zeros
        df = pd.DataFrame(0.0, index=[0], columns=expected_cols)
        
        # Fill numerical values
        for col, val in raw_dict.items():
            if col in df.columns:
                df.at[0, col] = val
                
        # 6. Fill One-Hot Encoded categorical values
        categories = {
            'gender': data.gender,
            'ever_married': data.ever_married,
            'work_type': data.work_type,
            'Residence_type': data.Residence_type,
            'smoking_status': data.smoking_status,
            'bmi_category': bmi_cat,
            'age_group': age_group
        }
        
        for cat_feature, val in categories.items():
            col_name = f"{cat_feature}_{val}"
            if col_name in df.columns:
                df.at[0, col_name] = 1
                
        return df

    @staticmethod
    def prepare_heart_data(data: PatientData, expected_cols: list) -> pd.DataFrame:
        # Categorize Age Risk
        if data.age <= 45: age_risk_tier = 0
        elif data.age <= 55: age_risk_tier = 1
        elif data.age <= 65: age_risk_tier = 2
        else: age_risk_tier = 3
            
        raw_dict = {
            'age': data.age,
            'sex': 1 if data.gender == 'Male' else 0, 
            'chest_pain_type': data.chest_pain_type,
            'exercise_angina': data.exercise_angina,
            'cp_severity': data.chest_pain_type, 
            'age_x_exang': data.age * data.exercise_angina,
            'age_risk_tier': age_risk_tier
        }
        
        df = pd.DataFrame(0.0, index=[0], columns=expected_cols)
        
        for col, val in raw_dict.items():
            if col in df.columns:
                df.at[0, col] = val
                
        return df

print(" Feature Engineering Engine initialized.")

In [ ]:
# FastAPI Application & Triage Logic

app = FastAPI(title="Unified Medical Triage API", version="1.0")

@app.post("/triage")
async def process_triage(patient: PatientData):
    try:
        
        # 1. Stroke Processing Pipeline
        
        # Engineer features
        df_stroke = FeatureEngineer.prepare_stroke_data(patient, stroke_features)
        
        # Scale data
        stroke_scaled = stroke_scaler.transform(df_stroke)
        
        # Predict Base Model
        stroke_prob = float(stroke_model.predict_proba(stroke_scaled)[0, 1])
        stroke_status = "Urgent (Red Code)" if stroke_prob >= 0.50 else "Stable (Green Code)"

        
        # 2. Heart Disease Processing Pipeline
        
        # Engineer features
        df_heart = FeatureEngineer.prepare_heart_data(patient, heart_features)
        
        # Scale data
        heart_scaled = heart_scaler.transform(df_heart)
        
        # Predict Stacking Ensemble (L0 then Meta)
        probs_rf = heart_l0['RF'].predict_proba(heart_scaled)[:, 1]
        probs_xgb = heart_l0['XGB'].predict_proba(heart_scaled)[:, 1]
        probs_lr = heart_l0['LR'].predict_proba(heart_scaled)[:, 1]
        
        meta_features = np.column_stack((probs_rf, probs_xgb, probs_lr))
        heart_prob = float(heart_meta.predict_proba(meta_features)[0, 1])
        heart_status = "Urgent (Red Code)" if heart_prob >= 0.50 else "Stable (Green Code)"

        
        # 3. Unified Decision Logic
        
        is_urgent = (stroke_prob >= 0.50) or (heart_prob >= 0.50)
        overall_status = "EMERGENCY" if is_urgent else "STABLE"
        print("===== AI TRIAGE RESULT =====")
        print(f"Stroke Probability : {stroke_prob:.4f}")
        print(f"Heart Probability  : {heart_prob:.4f}")
        print(f"Overall Status     : {overall_status}")
        print("============================")
         
        
        return {
            "overall_status": overall_status,
            "details": {
                "stroke_assessment": {
                    "probability": round(stroke_prob, 4),
                    "status": stroke_status
                },
                "heart_assessment": {
                    "probability": round(heart_prob, 4),
                    "status": heart_status
                }
            },
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("FastAPI application and routing logic configured.")

In [ ]:
# Production Server Execution (Permanent)
import uvicorn

print(" Starting Unified Medical Triage API...")
print(" Server is ready and listening on: http://127.0.0.1:8000")
print(" To stop the server, click the 'Interrupt' (Stop) button in Jupyter.")

uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")